# Implementing `Co-Clustering Triples from Open Information Extraction` From Scratch

### Setting Up Library

In [11]:
import torch
import random
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from itertools import combinations
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.cluster import AgglomerativeClustering

In [12]:
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Setup complete. Using device: {device}")

Setup complete. Using device: cpu


### 1. Data Simulation and Embedding Generation

In [13]:
def simulate_numerical_data():
    base_facts = [
        {'id': 1, 's': 'India', 'p': 'has a population of', 'o': 1400000000},
        {'id': 2, 's': 'Mount Everest', 'p': 'has a height of', 'o': 8848},
    ] # Simulated numerical data
    
    generated_facts = list(base_facts)
    aliases = {
        'India': ['Bharat', 'Hindustan'],
        'Mount Everest': ['Sagarmatha', 'Everest'],
    }
    pred_phrases = {'has a population of': 'population stands at',
                    'has a height of': 'is tall'}# predicted phrases
    for fact in base_facts:
        new_fact = fact.copy()# copying the fact
        new_fact['s'] = aliases.get(fact['s'], [fact['s']]) # randomly selecting an alias
        new_fact['p'] = pred_phrases.get(fact['p'], fact['p']) # using predicted phrase
        variation = fact['o'] * random.uniform(-0.1, 0.1) # adding variation to the object for numerical data
        new_fact['o'] = int(fact['o'] + variation) # updating the object with variation
        generated_facts.append(new_fact) # appending the new fact to the list
    generated_facts.extend([
        {'id': 3, 's': 'Brazil', 'p': 'has a GDP of', 'o': 1600000000000},
        {'id': 4, 's': 'K2', 'p': 'is located in', 'o': 8611},
    ])# adding more facts
    print("Simulated numerical dataset.")
    return generated_facts
        

- #### Building Embedding Generation

In [14]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')# loading the tokenizer
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)# loading the BERT model
BERT_DIM = bert_model.config.hidden_size # getting the BERT dimension

In [15]:
# function to get BERT embeddings for a list of texts
def get_bert_embedding(text_list):
    inputs = tokenizer(text_list, return_tensors='pt', padding=True, truncation=True, max_length=32).to(device)# tokenizing the input texts
    with torch.no_grad():
        outputs = bert_model(**inputs) # getting the BERT model outputs
    return outputs.last_hidden_state[:, 0, :].numpy() # returning the embeddings of the [CLS] token

- #### Implementing DICE Embeddings

In [16]:
class DICEEmbedder:
    def __init__(self, dimensions=100, max_val = 1e15):
        self.dimensions = dimensions# setting the dimensions for the embeddings
        self.max_val = max_val# setting the maximum value for the embeddings
    
    def get_dice_embedding(self, number):
        if not isinstance(number, (int, float)):
            return torch.zeros(1, self.dimensions) # returning zero vector for non-numerical inputs
        number = float(number) # converting the number to float
        embedding = torch.zeros(self.dimensions) # initializing the embedding vector
        scaled_num = torch.log(torch.tensor(abs(number)+ 1.0))# scaling the number
        for i in range(self.dimensions// 2):
            div_term = torch.exp(torch.tensor(i * -np.log(torch.tensor(self.max_val)) / (self.dimensions / 2)))# calculating the division term
            embedding[2*i] = torch.sin(scaled_num * div_term) # sine component
            embedding[2*i + 1] = torch.cos(scaled_num * div_term) # cosine component
        if number < 0:
            embedding[-1] = -1.0 # setting the last component to -1 for negative numbers
        return embedding.unsqueeze(0) # returning the embedding as a 2D tensor

In [17]:
# Initializing DICE embedder
DICE_DIM = 100 # setting the DICE dimensions
dice_embedder = DICEEmbedder(dimensions=DICE_DIM) # creating an instance of DICEEmbedder
print("DICE And BERT embeddings initialized with dimensions:", DICE_DIM, "and", BERT_DIM)

DICE And BERT embeddings initialized with dimensions: 100 and 768


### 2. Building PyTorch Model Architecture

In [18]:
# building primary model from paper for numerical data
class Model3_NM(nn.Module):
    
    def __init__(self):
        super(Model3_NM, self).__init__()# initializing the model
        self.relu = nn.ReLU() # ReLU activation function
        
        # building compoonent projection layers
        self.dense_proj_bert = nn.Linear(BERT_DIM, 128) # BERT projection layer
        self.mlp_dice = nn.Linear(DICE_DIM, 32) # DICE projection layer
        
        # Path 1st: Component wise similarity
        self.mlp_e = nn.Linear(256, 64)  # 128 * 2 i.e. BERT + DICE
        self.mlp_p = nn.Linear(256, 64)  # 128 * 2
        self.mlp_o = nn.Linear(320, 64)  # (128+32) * 2
        self.fusion_original = nn.Linear(192, 64)  # 64 * 3
        self.output_original = nn.Linear(64, 1)# final output layer for original embeddings
        
        #building path 2nd: fact-level similarity
        self.mlp_f = nn.Linear(288, 128) # 128(s) + 128(p) + 32(o_dice)
        self.fusion_final = nn.Linear(256, 64) # 128 * 2
        self.output_final = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()# final activation function
    
    # helper function to get fact representation
    def get_fact_rep(self, s_bert, p_bert, o_bert, o_dice):
        ds = self.relu(self.dense_proj_bert(s_bert))# processing subject BERT embeddings
        dp = self.relu(self.dense_proj_bert(p_bert))# processing predicate BERT embeddings
        do_bert = self.relu(self.dense_proj_bert(o_bert))# processing object BERT embeddings
        do_dice = self.relu(self.mlp_dice(o_dice))# processing object DICE embeddings
        
        # combining BERT and DICE embeddings
        combined_o = torch.cat((do_bert, do_dice), dim=1)
        
        fact_rep = self.relu(self.mlp_f(torch.cat([ds, dp, do_dice], dim=1))) # getting the fact representation
        return ds, dp, combined_o, fact_rep # returning the processed embeddings and fact representation
    
    
    def forward(self, s1_bert, p1_bert, o1_bert, s2_bert, p2_bert, o2_bert, o1_dice, o2_dice):
        ds1, dp1, co1, f1_rep = self.get_fact_rep(s1_bert, p1_bert, o1_bert, o1_dice)# getting fact representation for first fact
        ds2, dp2, co2, f2_rep = self.get_fact_rep( s2_bert, p2_bert, o2_bert, o2_dice)# getting fact representation for second fact
        
        # calculating component-wise similarities
        me = self.relu(self.mlp_e(torch.cat([ds1, ds2], dim=1)))# subject similarity
        mp = self.relu(self.mlp_p(torch.cat([dp1, dp2], dim=1)))# predicate similarity
        mo = self.relu(self.mlp_o(torch.cat([co1, co2], dim=1)))# object similarity
        
        fusion_orig_out = self.relu(self.fusion_original(torch.cat([me, mp, mo], dim=1)))# combining component-wise similarities
        out_orig = self.sigmoid(self.output_original(fusion_orig_out))# final output for original embeddings
        
        # calculating fact-level similarity
        fusion_final_out = self.relu(self.fusion_final(torch.cat([f1_rep, f2_rep], dim=1)))# combining fact representations
        out_final = self.sigmoid(self.output_final(fusion_final_out))# final output for fact-level similarity
        
        return out_orig, out_final


print("Defined Model 3 (N-M) architecture.")
        

Defined Model 3 (N-M) architecture.


### 3. Training And Handling Data

In [19]:
# dataset class for creating a dataset for fact pairs with BERT and DICE embeddings
class FactPairDataset(Dataset):
    def __init__(self, facts):
        self.pairs = []# initializing the dataset with fact pairs
        self.labels = []# initializing the labels for the pairs
        
        for fact1, fact2 in combinations(facts, 2):# creating pairs of facts
            self.pairs.append((fact1, fact2))# appending the pair to the dataset
            self.labels.append(1 if fact1['id'] == fact2['id'] else 0) # appending the label for the pair
    
    def __len__(self):
        return len(self.pairs) # returning the length of the dataset
    
    def __getitem__(self, idx):
        fact1, fact2 = self.pairs[idx] # getting the pair of facts
        label = torch.tensor(self.labels[idx], dtype = torch.float32) # getting the label for the pair
        return {
            's1': fact1['s'], 'p1': fact1['p'], 'o1_text': str(fact1['o']), 'o1_num': fact1['o'],
            's2': fact2['s'], 'p2': fact2['p'], 'o2_text': str(fact2['o']), 'o2_num': fact2['o'],
            'label': label
        }
            

In [20]:
def collate_fn(batch):
    # extracting subject, predicate, and object texts for both facts in the batch
    s1_texts = [item['s1'] for item in batch]
    p1_texts = [item['p1'] for item in batch]
    o1_texts = [item['o1_text'] for item in batch]
    s2_texts = [item['s2'] for item in batch]
    p2_texts = [item['p2'] for item in batch]
    o2_texts = [item['o2_text'] for item in batch]
    
    # batch BERT embeddings
    s1_bert = get_bert_embedding(s1_texts)
    p1_bert = get_bert_embedding(p1_texts)
    o1_bert = get_bert_embedding(o1_texts)
    s2_bert = get_bert_embedding(s2_texts)
    p2_bert = get_bert_embedding(p2_texts)
    o2_bert = get_bert_embedding(o2_texts)
    
    # Batch DICE embeddings
    o1_dice = torch.cat([dice_embedder.get_dice_embedding(item['o1_num']) for item in batch])
    o2_dice = torch.cat([dice_embedder.get_dice_embedding(item['o2_num']) for item in batch])
    labels = torch.stack([item['label'] for item in batch])# extracting labels from the batch

    return (s1_bert, p1_bert, o1_bert, s2_bert, p2_bert, o2_bert, o1_dice, o2_dice), labels